In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### 1. Data Loading and Preparation
In this step, we load the raw dataset and prepare it for machine learning.
* **Cleaning:** We remove duplicate rows (`drop_duplicates`) to prevent the model from becoming biased towards repeated patterns.
* **Class Distribution:** We print `value_counts()` to see how many samples we have for each category.
* **Splitting:** We divide the data into a **Training Set** (to teach the model) and a **Testing Set** (to evaluate it on unseen data).
* **Stratification:** Using `stratify=y` is crucial. It ensures that the exact same proportion of categories is maintained in both the training and testing sets, preventing imbalanced training.

In [6]:
test_data_percent = 20 # @param {"type":"slider","min":0,"max":100,"step":1}
df = pd.read_csv('https://github.com/Martin-Pop/traffic-classifier/raw/refs/heads/master/data.csv')
df.drop_duplicates(inplace=True)

print(df['category'].value_counts())

X = df.drop('category', axis=1)
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_percent/100, random_state=1, stratify=y)
# stratify=y same proportions



category
video       864
gaming      552
idle        463
browsing    424
voice       162
download    108
Name: count, dtype: int64


### 2. Model Training Pipeline
We use a `scikit-learn` Pipeline to chain data processing and model training together.
* **StandardScaler:** Standardizes features by removing the mean and scaling to unit variance.
* **Random Forest Classifier:** Learning method that builds multiple decision trees and merges them together.
    * `n_estimators`: The number of trees in the forest.
    * `max_depth`: Limits how deep the trees can grow to prevent overfitting.
    * `class_weight='balanced'`: Automatically adjusts weights. This is becouse some categories have much fewer samples than others.

In [7]:
n_estimators = 100 # @param {"type":"slider","min":0,"max":500,"step":5}
max_depth = 15 # @param {"type":"slider","min":0,"max":100,"step":1}
balanced = True # @param {"type":"boolean"}
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight='balanced' if balanced else None,
        random_state=1,
    ))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=15,
                                        random_state=67))])

### 3. Model Evaluation
Before saving the model, we must verify its performance on the unseen Testing Set.
* **Classification Report:** Displays key metrics like Precision, Recall, and F1-score for every single category. This tells us exactly where the model struggles and where it excels.
* **Feature Importances:** We extract the internal logic of the Random Forest to see which specific features were the most critical for the AI when deciding between categories.

In [8]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

rf_model = pipeline.named_steps['classifier']
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

print(feature_importances)

              precision    recall  f1-score   support

    browsing       0.87      0.86      0.86        85
    download       1.00      1.00      1.00        22
      gaming       0.99      1.00      1.00       110
        idle       0.87      0.95      0.91        93
       video       0.98      0.94      0.96       173
       voice       1.00      1.00      1.00        32

    accuracy                           0.95       515
   macro avg       0.95      0.96      0.95       515
weighted avg       0.95      0.95      0.95       515

size_mean          0.136626
size_std           0.136340
iat_mean           0.135207
in_byte_ratio      0.130086
iat_std            0.126818
iat_max            0.124595
tcp_ratio          0.051161
in_packet_ratio    0.049961
udp_ratio          0.047203
size_min           0.032126
size_max           0.029877
dtype: float64


### 4. Export Model for Production
Once satisfied with the model's accuracy, we serialize the entire pipeline (including the Scaler and the trained Random Forest) into a single `.joblib` file. This file can now be downloaded and directly integrated into our application.

In [ ]:
import joblib

joblib.dump(pipeline, 'model.joblib')

from google.colab import files
files.download('model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>